# Stage 3 — ESCO Occupation Classification (Batch API)
## Notebook 3.6 of 7 — Submit and Monitor Batch Jobs for Missing Records

**Role in the pipeline:**
Mirrors notebook 3.2, but operates on the missing-records batch files created in notebook 3.5. Submits JSONL files from `STAGE3_INPUT_PATH/missing/` as new Batch API jobs, then polls for completion and downloads output files to `STAGE3_OUTPUT_PATH/missing/`.

**⚠️ Asynchronous step — must be run twice:**
- **First run:** executes section 3.2.3 to submit missing-record jobs.
- **Second run (after Batch API completes, typically within 24 hours):** re-run section 3.2.4 to poll status and download output files.

**Position in Stage 3 sequence:**
1. 3.1 — Create batch input files
2. 3.2 — Submit batch jobs and monitor status
3. 3.3 — Extract classification results
4. 3.4 — Split missing and complete classifications
5. 3.5 — Create batch input files for missing records
6. ▶ **3.6 — Submit batch jobs for missing records** ← *you are here*
7. 3.7 — Extract results for missing records

**Prerequisites:**
- Notebook 3.5 completed — missing JSONL batch files exist
- `OPENAI_API_KEY` set in `.env`

**Documentation:** [Notebook guide](README.md) · [Stage data description](../../data/data-pipeline/stage_03/README_data.md)


## 3.2.1. Load packages and configuration

Loads standard libraries and the shared `general` config module.

In [ ]:
%load_ext autoreload
%autoreload 2
import sys
import os
from pipeline_bootstrap import configure_pipeline
configure_pipeline()
import general as g
g.clean_memory()

Imports Stage 2/3 processing modules, the OpenAI client, and JSON utilities.

In [ ]:
import stage2 as st2
import stage3 as st3
from openai import OpenAI
import json
import pandas as pd

## 3.2.2. Load processing logs

Reads the current tracker. The commented-out lines show how to reset the missing-record job columns before a retry run.

In [ ]:
process_df = pd.read_pickle(g.Config.STAGE3_PROCESS_PATH)
process_df.tail()

## 3.2.3. Create Batch Jobs

Initialises the OpenAI client.

In [ ]:
client = OpenAI(api_key=g.Config.OPENAI_API_KEY)

**Note:** You cannot submit all missing jobs simultaneously due to Batch API concurrency limits. Adjust `_from`/`_to` to submit in batches.

Local helper that reads the model name from the first line of a JSONL batch file — used for diagnostic output in the polling loop.

In [ ]:
def get_model_from_jsonl(path):
    with open(path, "r", encoding="utf-8") as f:
        first_line = f.readline()
    data = json.loads(first_line)
    return data["body"]["model"]

Missing-record job submission loop: iterates over `process_df[_from:_to]`. Rows with `missing_input_batch_status == 'empty'` are skipped. Rows already with a `completed`/`in_progress`/`finalizing`/`validating` job status are skipped. For eligible rows, `create_batch_job()` uploads the JSONL and registers the job ID.

In [ ]:
_from = 0
_to = 1616

process_df = pd.read_pickle(g.Config.STAGE3_PROCESS_PATH)
#
for _,row in process_df[_from:_to].iterrows():
    if row["missing_input_batch_status"] == "empty":
        continue
    if row["missing_input_batch_status"] == "created":
        if row["missing_job_status"] == "completed" or row["missing_job_status"] == "in_progress" or row["missing_job_status"] == "finalizing" or row["missing_job_status"] == "validating":
            continue
        else:
            batch_job = st3.create_batch_job(client, row["missing_input_batch_path"])
            process_df.loc[_, "missing_job_id"] = batch_job.id
            process_df.loc[_, "missing_job_status"] = "in_progress"
            process_df.to_pickle(g.Config.STAGE3_PROCESS_PATH)
            print("Added job for: ", str(_), "-", row["input_file"])

## 3.2.4. Check Jobs and Download Files

**Polling and download loop for missing records** — run after the Batch API completes. For each row with `missing_job_status == 'in_progress'` (or `finalizing`/`validating`):

1. Retrieves job status from the API.
2. If `completed`: downloads output JSONL to `STAGE3_OUTPUT_PATH/missing/`, sets `missing_output_batch_status = 'complete'`.
3. If still running: prints progress.
4. If error: prints status and row index.

The tracker is saved after every row update.

In [ ]:
process_df = pd.read_pickle(g.Config.STAGE3_PROCESS_PATH)
index = 0
#[_from:_to]
for _,row in process_df.iterrows():

    if row["missing_job_status"] == "in_progress" or row["missing_job_status"] == "finalizing" or row["missing_job_status"] == "validating":
        index += 1
        try:
            batch = client.batches.retrieve(row["missing_job_id"])
            print("row:", _, "-",batch.status, "-", row["missing_job_id"])
        except ValueError as e:
            process_df.loc[_, "missing_job_status"] = None
            process_df.loc[_, "missing_job_id"] = None

        process_df.loc[_, "missing_job_status"] = batch.status

        if batch.status == "completed":
            result_file_id = batch.output_file_id
            result = client.files.content(result_file_id).content
            output_path = os.path.join(g.Config.STAGE3_OUTPUT_PATH, "missing", process_df.loc[_, "input_file"]  + ".json")
            with open(output_path, 'wb') as file:
                file.write(result)
            process_df.loc[_, "missing_output_batch_path"] = output_path
            process_df.loc[_, "missing_output_batch_status"] = "complete"
            print(index, "Added completed batch file for: ", row["input_file"])
        elif batch.status != "in_progress" and batch.status != "finalizing":
            print(index, "Error: ", row["input_file"], " Status: ", batch.status, "-", get_model_from_jsonl(row["missing_input_batch_path"]), " - ", batch.request_counts.total,  " rows, ", f"row({_})")
        else:
            print(index, "Waiting for: ", row["input_file"], " Status: ", batch.status, "-", batch.request_counts.completed, "of", batch.request_counts.total, "-", st3.get_model_from_jsonl(row["missing_input_batch_path"]), f"row ({_})")

process_df.to_pickle(g.Config.STAGE3_PROCESS_PATH)

---
© 2026 Yurii Kleban, Britta Rude. All rights reserved.